# TraceQual: Decision Matrix from a Synthetic Coding Session

TraceQual is a disclosure scaffold for qualitative researchers whose analysis involves AI. It reads a researcher-AI chat transcript and produces a structured matrix documenting each analytic decision the researcher made about an AI suggestion, intended to be inspectable in an appendix. This notebook renders one locked extraction run on a synthetic fixture. TraceQual is not a tool for AI coding; it is a tool for disclosing AI involvement that is already happening.

## Provenance

This notebook reads cached output only; it does not re-run extraction.

| Field | Value |
|---|---|
| Model | `claude-sonnet-4-6` |
| Temperature | `0.0` |
| Prompt version | `1.0.0` (re-locked 2026-05-25; see `prompts/prompt_changelog.md`) |
| Schema version | `1.0.0` (model-reported `confidence`; predates schema v1.1.0 boolean flags) |
| Fixture | `docs/synthetic_chat_long.md` (60-turn synthetic coding session) |
| Cache file | `outputs/synthetic_long_decisions__prompt-1.0.0.json` |
| Run timestamp | `2026-05-24` (from cache filename suffix and file date) |

In [1]:
import json
from pathlib import Path

import pandas as pd

REPO_ROOT = Path("..").resolve()
CACHE_PATH = REPO_ROOT / "outputs" / "synthetic_long_decisions__prompt-1.0.0.json"

with CACHE_PATH.open(encoding="utf-8") as handle:
    rows = json.load(handle)

df = pd.DataFrame(rows)

print(f"Row count: {len(df)}")
print()
print("Decision value distribution:")
print(df["decision"].value_counts().to_string())
print()
print("Confidence value distribution:")
print(df["confidence"].value_counts().to_string())

Row count: 23

Decision value distribution:
decision
accepted    11
modified     6
rejected     3
deferred     2
unclear      1

Confidence value distribution:
confidence
high      19
medium     3
low        1


## Schema walkthrough

**`turn_id`** anchors each row to the AI turn that made the substantive suggestion. Turn order, not wall-clock time, defines analytic proximity in pasted transcripts where timestamps are often absent.

**`timestamp`** records ISO 8601 datetime when the export format provides it (Claude JSON exports); it is null for pasted fixtures. Feuston and Brubaker (2021) motivate stage-aware documentation rather than session-level timestamps, but long pauses can signal reflective breaks in a future positioning pass.

**`researcher_prompt_summary`** is a one-sentence neutral summary of what the researcher asked. Summaries reduce appendix bulk and limit re-circulation of participant data; the tradeoff is lost nuance, which the original transcript must retain for audit.

**`ai_suggestion_summary`** is a one-sentence neutral summary of what the AI proposed. Paired with the prompt summary, it makes each row interpretable without re-reading the full chat.

**`decision`** captures the researcher's response using five minimally overlapping values: `accepted`, `modified`, `rejected`, `deferred`, and `unclear`. The `deferred` category preserves a practice Feuston and Brubaker (2021) document, where researchers treat AI output as ideas to revisit rather than immediate inputs.

**`reasoning`** holds the researcher's stated or inferred rationale. Inferred reasoning carries an `[inferred]` prefix so readers can distinguish transcript evidence from extractor inference, aligning with McDonald et al. (2019) on heterogeneous reliability practices: the matrix is an audit trail, not an IRR report.

**`analytic_stage`** maps each row to Braun and Clarke's six phases (`familiarization`, `coding`, `theming`, `reviewing`, `defining`, `writeup`). A closed enum supports cross-study interpretability; `coding` is the default when the stage is ambiguous.

**`confidence`** in this locked v1.0.0 run is model-reported. Schema v1.1.0 later replaced self-report with rule-derived confidence from `decision_stated` and `reasoning_stated` booleans; see `docs/schema_notes.md` for the full rationale.

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", None)

df

,turn_id,timestamp,researcher_prompt_summary,ai_suggestion_summary,decision,reasoning,analytic_stage,confidence
0,6,None,The researcher pasted transcript P5 and asked the AI to begin coding it agai...,The AI suggested that 'Tool as buffer' (Code 3) may not fire in P5 because P...,rejected,"[inferred] The researcher later accepted a Code 3 assignment at turn 19, and...",familiarization,medium
1,6,None,The researcher pasted transcript P5 and asked the AI to begin coding it agai...,The AI observed that the re-recording passage around the 30-minute mark fits...,rejected,[inferred] The researcher subsequently coded the re-recording passage straig...,familiarization,medium
2,8,None,The researcher asked the AI to proceed excerpt by excerpt from the top of th...,The AI proposed coding lines 12-19 as Code 2 (Asynchronous control) with a s...,modified,The researcher accepted the Code 2 assignment but rejected the supporting no...,coding,high
3,10,None,The researcher accepted Code 2 for lines 12-19 and indicated they would writ...,"The AI proposed coding lines 34-41 as Code 6 (Re-do compulsion), describing ...",accepted,The researcher accepted without qualification.,coding,high
4,12,None,The researcher accepted Code 6 for lines 34-41.,The AI proposed that lines 52-58 fit both Code 4 (Manager visibility anxiety...,modified,The researcher accepted both codes but specified Code 4 as primary and Code ...,coding,high
5,14,None,The researcher assigned Code 4 primary and Code 5 secondary to lines 52-58.,The AI noted that lines 71-79 do not fit any current code and offered the re...,accepted,The researcher chose to flag the excerpt and directed the AI to add a candid...,coding,high
6,16,None,The researcher directed the AI to add a candidate code 'Skill development [?...,The AI proposed that lines 89-95 fit Code 7 (Time displacement) but also sur...,deferred,The researcher explicitly set the excerpt aside for later consideration with...,coding,high
7,18,None,The researcher deferred the decision on lines 89-95 and asked the AI to move...,"The AI proposed coding lines 102-110 as Code 3 (Tool as buffer), retracting ...",accepted,The researcher accepted the Code 3 assignment and acknowledged the AI's self...,coding,high
8,20,None,The researcher accepted Code 3 for lines 102-110.,The AI proposed coding lines 118-125 as Code 4 (Manager visibility anxiety) ...,accepted,The researcher accepted Code 4 and chose to hold the visibility-as-currency ...,coding,high
9,20,None,The researcher accepted Code 4 for lines 118-125.,The AI implicitly offered 'visibility as currency' as a possible analytic co...,deferred,The researcher found the construct interesting but explicitly chose to hold ...,coding,high


## Inclusion-bias artifacts

Turns 27 and 47 are documented v1 failure modes where the extractor emitted rows for exchanges adjacent to turns that should have been skipped. Turn 27 follows the administrative cross-transcript digression (turns 24-26), where the researcher asked the AI to read transcripts not in context. Turn 47 follows the explicitly skipped out-of-scope excerpt (turns 45-46). The harness checks `skip_turns_24_26` and `skip_turns_45_46` use a +/-1 turn tolerance window, so rows anchored at turns 27 and 47 register as failures even though the rows themselves describe substantive coding suggestions.

In [3]:
artifact_rows = df[df["turn_id"].isin([27, 47])]
artifact_rows

,turn_id,timestamp,researcher_prompt_summary,ai_suggestion_summary,decision,reasoning,analytic_stage,confidence
11,27,None,After a digression about cross-transcript comparison that could not be fulfi...,The AI proposed coding lines 155-164 under the 'Skill development' candidate...,modified,The researcher rejected the Skill development framing as too positive and di...,coding,high
19,47,None,The researcher agreed to skip the out-of-scope lines 308-315 and asked the A...,The AI proposed that lines 321-330 do not fit the current codebook directly ...,accepted,The researcher accepted and supplied the candidate code label 'Norm dependen...,coding,high


## Validation harness results

Full output from the locked v1.0.0 run (`scripts/validate_extraction.py`, 12 PASS / 4 FAIL / 3 PENDING / 4 N/A):

```
TraceQual Extraction Validation Report
Fixture: docs/synthetic_chat_long.md
Model: claude-sonnet-4-6 | Temperature: 0.0 | Prompt: 1.0.0 | Schema: 1.0.0
Cache: outputs/synthetic_long_decisions__prompt-1.0.0.json
Turn anchor: matrix rows use AI turn_id (+/-1 tolerance for fixture crosswalk)
Run: 2026-05-25T00:00:00Z

[FAIL]    matrix_row_count                23 rows (expected 18-22)
[PASS]    decision_enum_accepted          Found 6 'accepted' rows near expected turns (expected >=5)
[PASS]    decision_enum_modified          Found 2 'modified' rows near turns 8, 27 (expected >=2)
[PASS]    decision_enum_rejected          Found 1 'rejected' row(s) near turns 35-36
[PASS]    decision_enum_deferred          Found 1 'deferred' row(s) near turns 16-17, 37-38
[PASS]    decision_enum_unclear           Found 1 'unclear' row(s) near turns 22-23
[PASS]    decision_enum_coverage          All five decision enum values present
[PASS]    analytic_stage_coding_dominant  20/23 rows tagged 'coding' (majority)
[PASS]    analytic_stage_theming_shift    Theming detected at turn_id(s) [39] (expected >= turn 39)
[FAIL]    skip_turns_24_26                Unexpected row(s) at turn_id [27] (administrative exchange)
[FAIL]    skip_turns_45_46                Unexpected row(s) at turn_id [47] (out-of-scope skip)
[PASS]    skip_turns_58_59                No rows at turn_id 58 or 59
[PASS]    skip_correction_55_57           No row documents the turn 55-57 miscount correction
[PASS]    low_confidence_t36              Row near turn 36 with confidence medium
[FAIL]    low_confidence_t38              No low confidence row near turn 38
[PASS]    low_confidence_t22_23           Low confidence or unclear decision near turns 22-23
[N/A]     confidence_distribution         Descriptive only: high=19, medium=3, low=1
[N/A]     confidence_high_correct_count   Descriptive only: 9/13 high-confidence rows are fixture-correct (0.692)
[N/A]     confidence_low_wrong_count      Descriptive only: 0/1 low-confidence rows are fixture-wrong (0.000)
[N/A]     confidence_calibration_signal   Insufficient data for calibration verdict: high n=13, low n=1, minimum per bucket=5
[PENDING] positioning_reframing_t18       Positioning pass not yet implemented. See docs/project_structure.md and docs/schema_notes.md (Secondary table: positioning_log).
[PENDING] positioning_broadening_t32      Positioning pass not yet implemented. See docs/project_structure.md and docs/schema_notes.md (Secondary table: positioning_log).
[PENDING] positioning_reframing_t40       Positioning pass not yet implemented. See docs/project_structure.md and docs/schema_notes.md (Secondary table: positioning_log).

---
23 checks: 12 PASS, 4 FAIL, 3 PENDING, 4 N/A
Pending implementation: positioning pass (tracequal/positioning.py)
Exit code: 1
```

**FAIL annotations (one sentence each):**

- **`matrix_row_count`**: The extractor produced 23 rows, one above the fixture's documented 18-22 range, likely from splitting multi-suggestion AI turns at turn 6.
- **`skip_turns_24_26`**: A row at turn 27 falls inside the +/-1 tolerance window around the administrative exchange at turns 24-26.
- **`skip_turns_45_46`**: A row at turn 47 falls inside the tolerance window around the out-of-scope skip at turns 45-46.
- **`low_confidence_t38`**: The fixture expects a low-confidence or deferred row near turn 38 where the researcher skipped without explanation; the extractor assigned `rejected` with high confidence instead.

**PENDING annotations (one sentence each):**

- **`positioning_reframing_t18`**: The secondary positioning pass that should detect the AI's self-correction on Code 3 at turn 18 is not yet implemented.
- **`positioning_broadening_t32`**: The expected broadening shift when the researcher expands Code 3 at turn 32 awaits the positioning module.
- **`positioning_reframing_t40`**: The expected reframing when the researcher creates the "Performative async" theme at turn 40 also awaits implementation.

## Three findings

Prompt revisions are not monotonically improving. Prompt v1.1.0 targeted the skip-turn and rejection failures but produced 10 harness PASS checks versus 12 at the re-locked v1.0.0 baseline, and introduced new failures on `decision_enum_modified` and `low_confidence_t38`. Both prompt versions remain cached under `outputs/` for comparison.

Temperature-zero non-determinism is comparable to prompt effects. Two extractions at temperature 0.0 on the same fixture and prompt family produced 20-row and 23-row matrices with different decision distributions (`outputs/synthetic_long_decisions.json` versus `outputs/synthetic_long_decisions__prompt-1.0.0.json`), a row-count delta similar in magnitude to the 20 versus 23 split between prompt v1.1.0 and v1.0.0 caches.

The confidence signal collapses toward medium under rule-derived scoring. When schema v1.1.0 replaced model self-report with booleans (`decision_stated`, `reasoning_stated`) and Python-side derivation, roughly two-thirds of rows on the synthetic fixture landed at medium because researchers state decisions more often than they state reasons. On this locked v1.0.0 cache, model-reported confidence skews high (19 of 23 rows), which is itself a finding about self-report unreliability.

## Limitations and what is not in scope

This notebook uses a single synthetic fixture (`docs/synthetic_chat_long.md`), not real interview data. The epistemic positioning pass (`tracequal/positioning.py`) is unimplemented; three harness checks remain PENDING. Format handling is bounded: Claude export JSON is tested, ChatGPT tree exports are explicitly unsupported, and markdown paste is the primary recommendation for other tools. Validation is n-of-1: harness expectations are tuned to one fixture, and confidence calibration requires at least five rows per bucket before issuing a verdict.

## Closing

For the full methods analysis, see [docs/methods_reflection.md](../docs/methods_reflection.md). For redesign sketches (confidence rule revision, ChatGPT export support), see [docs/future_work.md](../docs/future_work.md).